In [47]:
import json
import numpy as np
import pandas as pd
import os
import sys

sys.path.append(os.path.join(os.getcwd(), ".."))

from system_ranking import rank_systems
import warnings

warnings.simplefilter(action="ignore", category=FutureWarning)

# Significance by LP

In [48]:
model2info = {
    "baseline": {
        "alias": "NLLB",
    },
    "greedy-7b-wo-context": {
        "alias": "TowerInstruct-7B w/o context",
    },
    "greedy-7b-w-context": {
        "alias": "TowerInstruct-7B w/ context",
    },
    "greedy-7b-5shot-wo-context": {
        "alias": "TowerInstruct-7B w/o context",
    },
    "greedy-7b-5shot-w-context": {
        "alias": "TowerInstruct-7B w/ context",
    },
    "greedy-70b-5shot-wo-context": {
        "alias": "TowerInstruct-70B w/o context",
    },
    "greedy-70b-5shot-w-context": {
        "alias": "TowerInstruct-70B w/ context",
    },
    "greedy-7b-ft-wo-context": {
        "alias": "TowerChat-7B w/o context",
    },
    "greedy-7b-ft-w-context": {
        "alias": "TowerChat-7B w/ context",
    },
    "mbr": {
        "alias": "TowerChat-7B w/ context + COMET MBR",
    },
    "mbr-source": {
        "alias": "TowerChat-7B w/ context + ContextCOMET MBR",
    },
}

models = list(model2info.keys())

In [49]:
def load_results(results_df, model, lp, metric):
    metric_score = results_df[results_df["lp"] == lp].loc[:, f"{model}-{metric}"].mean()
    return {
        "model": model,
        "lp": lp,
        metric: metric_score,
    }


def load_segment_results(results_df, model, lp, metric):
    return results_df[results_df["lp"] == lp].loc[:, f"{model}-{metric}"].tolist()

In [50]:
metric = "comet"
lps = [
    "en-de",
    "en-fr",
    "en-pt-br",
    "en-ko",
    "en-nl",
    "de-en",
    "fr-en",
    "pt-br-en",
    "ko-en",
    "nl-en",
]
lp_names = [
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
]
results = pd.concat(
    [
        pd.DataFrame(
            [
                load_results(
                    pd.read_csv(f"../paper_results/test.{lp_name}.csv"),
                    model,
                    lp,
                    metric,
                )
                for lp, lp_name in zip(lps, lp_names)
            ]
        )
        for model in models
    ]
)
pivot_results = results.pivot(columns="lp", index="model", values=metric)
pivot_results = pivot_results.loc[models]

clusters_by_lp = {}
threshold_p_value = 0.05
for lp, lp_name in zip(lps, lp_names):
    data = {
        m: load_segment_results(
            pd.read_csv(f"../paper_results/test.{lp_name}.csv"), m, lp, metric
        )
        for m in models
    }
    seg_scores = pd.DataFrame(data)  # for metricx, lower is better

    clusters = rank_systems(seg_scores, threshold_p_value=threshold_p_value)

    clusters_by_lp[lp] = clusters

clusters = pd.DataFrame(clusters_by_lp)
clusters

,en-de,en-fr,en-pt-br,en-ko,en-nl,de-en,fr-en,pt-br-en,ko-en,nl-en
baseline,7.0,6.0,9.0,9.0,8.0,6.0,6.0,8.0,9.0,8.0
greedy-7b-wo-context,5.0,5.0,7.0,5.0,7.0,4.0,2.0,7.0,6.0,5.0
greedy-7b-w-context,6.0,6.0,8.0,7.0,7.0,5.0,5.0,6.0,7.0,7.0
greedy-7b-5shot-wo-context,5.0,5.0,6.0,6.0,6.0,4.0,4.0,6.0,5.0,5.0
greedy-7b-5shot-w-context,6.0,6.0,8.0,8.0,6.0,4.0,5.0,5.0,8.0,6.0
greedy-70b-5shot-wo-context,2.0,4.0,5.0,4.0,5.0,1.0,1.0,4.0,6.0,3.0
greedy-70b-5shot-w-context,3.0,4.0,5.0,5.0,4.0,4.0,4.0,4.0,6.0,3.0
greedy-7b-ft-wo-context,4.0,4.0,4.0,3.0,4.0,3.0,2.0,5.0,4.0,4.0
greedy-7b-ft-w-context,2.0,3.0,3.0,2.0,3.0,3.0,3.0,3.0,3.0,3.0
mbr,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0


In [52]:
cluster_dict = {
    "1.0": "firstcluster",
    "2.0": "secondcluster",
    "3.0": "thirdcluster",
    "4.0": "fourthcluster",
    "5.0": "fifthcluster",
    "6.0": "sixthcluster",
    "7.0": "seventhcluster",
    "8.0": "eighthcluster",
    "9.0": "ninthcluster",
    "10.0": "tenthcluster",
}
# join with latex
latex_results = (pivot_results * 100).round(2)
# sort latex_results columns to match lps
latex_results = latex_results[lps]
for model in models:
    print(model, end=": ")
    for lp in lps:
        if lp == "en-nl":
            end = " && "
        elif lp == "nl-en":
            end = " \\\\"
        else:
            end = " & "
        print(
            f"{latex_results.loc[model, lp]}\\{cluster_dict[str(clusters.loc[model, lp])]}",
            end=end,
        )
    print("\n")

baseline: 90.56\seventhcluster & 91.06\sixthcluster & 86.33\ninthcluster & 87.26\ninthcluster & 87.86\eighthcluster && 89.03\sixthcluster & 89.18\sixthcluster & 86.1\eighthcluster & 88.05\ninthcluster & 88.45\eighthcluster \\

greedy-7b-wo-context: 91.71\fifthcluster & 91.89\fifthcluster & 91.9\seventhcluster & 91.64\fifthcluster & 91.3\seventhcluster && 92.08\fourthcluster & 92.78\secondcluster & 90.43\seventhcluster & 93.13\sixthcluster & 92.45\fifthcluster \\

greedy-7b-w-context: 91.48\sixthcluster & 91.08\sixthcluster & 90.79\eighthcluster & 91.13\seventhcluster & 91.0\seventhcluster && 91.33\fifthcluster & 91.89\fifthcluster & 90.63\sixthcluster & 91.88\seventhcluster & 91.08\seventhcluster \\

greedy-7b-5shot-wo-context: 91.75\fifthcluster & 91.75\fifthcluster & 92.32\sixthcluster & 91.41\sixthcluster & 91.55\sixthcluster && 92.06\fourthcluster & 92.28\fourthcluster & 90.63\sixthcluster & 93.55\fifthcluster & 92.6\fifthcluster \\

greedy-7b-5shot-w-context: 91.41\sixthcluster & 

# Aggregate significance

In [67]:
metrics = ["comet", "context-comet-qe", "metricx"]
lps = [
    "en-de",
    "en-fr",
    "en-pt-br",
    "en-ko",
    "en-nl",
    "de-en",
    "fr-en",
    "pt-br-en",
    "ko-en",
    "nl-en",
]
lp_names = [
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
]
from_en_lps = [lp for lp in lps if lp.startswith("en-")]
into_en_lps = [lp for lp in lps if lp.endswith("-en")]
for metric in metrics:
    print(metric)
    results = pd.concat(
        [
            pd.DataFrame(
                [
                    load_results(
                        pd.read_csv(f"../paper_results/test.{lp_name}.csv"),
                        model,
                        lp,
                        metric,
                    )
                    for lp, lp_name in zip(lps, lp_names)
                ]
            )
            for model in models
        ]
    )
    if metric == "metricx":
        results[metric] = -results[metric]
    clusters_by_lp = {}
    threshold_p_value = 0.05
    for lp, lp_name in zip(lps, lp_names):
        data = {
            m: load_segment_results(
                pd.read_csv(f"../paper_results/test.{lp_name}.csv"), m, lp, metric
            )
            for m in models
        }
        seg_scores = pd.DataFrame(data)
        if metric == "metricx":
            seg_scores = -seg_scores

        clusters = rank_systems(seg_scores, threshold_p_value=threshold_p_value)

        clusters_by_lp[lp] = clusters

    clusters = pd.DataFrame(clusters_by_lp)

    # aggregate and get bordas
    pivot_results = results.pivot(columns="lp", index="model", values=metric)
    pivot_results = pivot_results.loc[models]
    into_en_results = pivot_results.loc[:, into_en_lps].mean(axis=1)
    from_en_results = pivot_results.loc[:, from_en_lps].mean(axis=1)

    from_en_borda_count = (
        (clusters[from_en_lps].sum(axis=1).sort_values() / len(from_en_lps))
        .round(0)
        .loc[models]
    )
    into_en_borda_count = (
        (clusters[into_en_lps].sum(axis=1).sort_values() / len(into_en_lps))
        .round(0)
        .loc[models]
    )
    print(f"From EN borda counts")
    display(from_en_borda_count)
    print(f"Into EN borda counts")
    display(into_en_borda_count)

comet
From EN borda counts


baseline                       8.0
greedy-7b-wo-context           6.0
greedy-7b-w-context            7.0
greedy-7b-5shot-wo-context     6.0
greedy-7b-5shot-w-context      7.0
greedy-70b-5shot-wo-context    4.0
greedy-70b-5shot-w-context     4.0
greedy-7b-ft-wo-context        4.0
greedy-7b-ft-w-context         3.0
mbr                            1.0
mbr-source                     2.0
dtype: float64

Into EN borda counts


baseline                       7.0
greedy-7b-wo-context           5.0
greedy-7b-w-context            6.0
greedy-7b-5shot-wo-context     5.0
greedy-7b-5shot-w-context      6.0
greedy-70b-5shot-wo-context    3.0
greedy-70b-5shot-w-context     4.0
greedy-7b-ft-wo-context        4.0
greedy-7b-ft-w-context         3.0
mbr                            1.0
mbr-source                     2.0
dtype: float64

context-comet-qe
From EN borda counts


...

baseline                       6.0
greedy-7b-wo-context           4.0
greedy-7b-w-context            4.0
greedy-7b-5shot-wo-context     3.0
greedy-7b-5shot-w-context      4.0
greedy-70b-5shot-wo-context    1.0
greedy-70b-5shot-w-context     2.0
greedy-7b-ft-wo-context        3.0
greedy-7b-ft-w-context         3.0
mbr                            1.0
mbr-source                     1.0
dtype: float64

metricx
From EN borda counts


baseline                       6.0
greedy-7b-wo-context           3.0
greedy-7b-w-context            5.0
greedy-7b-5shot-wo-context     3.0
greedy-7b-5shot-w-context      5.0
greedy-70b-5shot-wo-context    2.0
greedy-70b-5shot-w-context     2.0
greedy-7b-ft-wo-context        3.0
greedy-7b-ft-w-context         3.0
mbr                            2.0
mbr-source                     2.0
dtype: float64

Into EN borda counts


baseline                       5.0
greedy-7b-wo-context           4.0
greedy-7b-w-context            4.0
greedy-7b-5shot-wo-context     4.0
greedy-7b-5shot-w-context      4.0
greedy-70b-5shot-wo-context    2.0
greedy-70b-5shot-w-context     2.0
greedy-7b-ft-wo-context        3.0
greedy-7b-ft-w-context         2.0
mbr                            1.0
mbr-source                     1.0
dtype: float64

### chrf particular case

In [86]:
from sacrebleu import corpus_chrf
from tqdm import tqdm

n_iters = 100
pct_dataset = 0.5


def load_results_chrf(results_df, model, lp, metric):
    lp_subset = results_df[results_df["lp"] == lp]
    references = [str(s) for s in lp_subset["reference"].tolist()]
    hypotheses = [str(s) for s in lp_subset[f"{model}"].tolist()]
    metric_score = corpus_chrf(hypotheses, [references]).score
    return {
        "model": model,
        "lp": lp,
        metric: metric_score,
    }


def load_segment_results_chrf(results_df, model, lp, n_iters, pct_dataset):
    lp_subset = results_df[results_df["lp"] == lp]
    scores = []
    for i in tqdm(range(n_iters), total=n_iters, desc=f"Bootstrapping chrf for {lp}"):
        # make sure we test on different subsets for different models
        random_subset = lp_subset.sample(frac=pct_dataset, random_state=i).reset_index(
            drop=True
        )
        references = [str(s) for s in random_subset["reference"].tolist()]
        hypotheses = [str(s) for s in random_subset[f"{model}"].tolist()]
        scores.append(corpus_chrf(hypotheses, [references]).score)

    return scores


lps = [
    "en-de",
    "en-fr",
    "en-pt-br",
    "en-ko",
    "en-nl",
    "de-en",
    "fr-en",
    "pt-br-en",
    "ko-en",
    "nl-en",
]
lp_names = [
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
    "en-de",
    "en-fr",
    "en-pt",
    "en-ko",
    "en-nl",
]
from_en_lps = [lp for lp in lps if lp.startswith("en-")]
into_en_lps = [lp for lp in lps if lp.endswith("-en")]
metric = "chrf"
results = pd.concat(
    [
        pd.DataFrame(
            [
                load_results_chrf(
                    pd.read_csv(f"../paper_results/test.{lp_name}.csv"),
                    model,
                    lp,
                    metric,
                )
                for lp, lp_name in zip(lps, lp_names)
            ]
        )
        for model in models
    ]
)
if metric == "metricx":
    results[metric] = -results[metric]
clusters_by_lp = {}
threshold_p_value = 0.05
for lp, lp_name in zip(lps, lp_names):
    data = {
        m: load_segment_results_chrf(
            pd.read_csv(f"../paper_results/test.{lp_name}.csv"),
            m,
            lp,
            n_iters,
            pct_dataset,
        )
        for m in models
    }
    seg_scores = pd.DataFrame(data)
    if metric == "metricx":
        seg_scores = -seg_scores

    clusters = rank_systems(seg_scores, threshold_p_value=threshold_p_value)

    clusters_by_lp[lp] = clusters

clusters = pd.DataFrame(clusters_by_lp)

# aggregate and get bordas
pivot_results = results.pivot(columns="lp", index="model", values=metric)
pivot_results = pivot_results.loc[models]
into_en_results = pivot_results.loc[:, into_en_lps].mean(axis=1)
from_en_results = pivot_results.loc[:, from_en_lps].mean(axis=1)

from_en_borda_count = (
    (clusters[from_en_lps].sum(axis=1).sort_values() / len(from_en_lps))
    .round(0)
    .loc[models]
)
into_en_borda_count = (
    (clusters[into_en_lps].sum(axis=1).sort_values() / len(into_en_lps))
    .round(0)
    .loc[models]
)
print(f"From EN borda counts")
display(from_en_borda_count)
print(f"Into EN borda counts")
display(into_en_borda_count)

From EN borda counts


baseline                       10.0
greedy-7b-wo-context            8.0
greedy-7b-w-context            10.0
greedy-7b-5shot-wo-context      8.0
greedy-7b-5shot-w-context       9.0
greedy-70b-5shot-wo-context     5.0
greedy-70b-5shot-w-context      6.0
greedy-7b-ft-wo-context         5.0
greedy-7b-ft-w-context          3.0
mbr                             2.0
mbr-source                      1.0
dtype: float64

Into EN borda counts


baseline                       10.0
greedy-7b-wo-context            6.0
greedy-7b-w-context             8.0
greedy-7b-5shot-wo-context      7.0
greedy-7b-5shot-w-context       9.0
greedy-70b-5shot-wo-context     4.0
greedy-70b-5shot-w-context      6.0
greedy-7b-ft-wo-context         3.0
greedy-7b-ft-w-context          2.0
mbr                             2.0
mbr-source                      2.0
dtype: float64